# DS3000 Project Proposal: Real Supernovae vs. False-Positive Transients

### Abigail Regan & Ashley Minuskin

*Data source: the Open Supernova Catalog (astrocatalogs project on GitHub)*

## 1. Project Description and Motivation

Robotic sky surveys such as the *Zwicky Transient Facility (ZTF)* and *ATLAS* scan large parts of the night sky every night, automatically flagging anything that appears or changes brightness, a 'transient'. Astronomers hope many of these transients are newly exploding stars, or supernovae, but in practice a large share of them turn out to be something else entirely: a flickering active galactic nucleus, a cataclysmic variable star, a nova, or even a passing asteroid. Because follow-up observations with large telescopes are expensive and limited, separating genuine supernova candidates from these false-positive transients as early and automatically as possible is a real, active problem in observational astronomy. Survey teams like ZTF have even built deep-learning 'real-bogus' classifiers specifically to solve it (Duev et al., 2019, [MNRAS](https://arxiv.org/abs/1907.11259)).

The Open Supernova Catalog (Guillochon, Parrent, Kelley, & Margutti, 2017, [ApJ](https://ar5iv.arxiv.org/html/1605.01054)) is a public, community-maintained archive that tracks both sides of this problem: a catalog of candidate and confirmed supernovae, and a 'boneyard' of transients that were initially suspected to be supernovae but were later reclassified as something else. This project uses that catalog to explore two questions:

1. **Can information available at the moment of discovery: how bright an object appeared, how many early brightness measurements it has, and which survey found it, help distinguish a real supernova candidate from a false-positive 'boneyard' object?**
2. **Do different discovery surveys (ZTF and ATLAS here) differ in how often the transients they report turn out to be real supernova candidates versus false positives? In other words, is one survey more prone to flagging impostors than the other?**

Answering these questions matters because even a simple, early classifier could help astronomers prioritize which of the thousands of nightly alerts are worth the limited, expensive follow-up investigation.

## 2. Data Source and Collection

### Where the data comes from

I've gathered data from the Open Supernova Catalog (Guillochon et al. 2017), an open collection of transient-event data maintained by the astrocatalogs project on GitHub. Because the full catalog is too large for a single repository, it is split into several GitHub repositories by discovery-date range. I will use two of them:

- [astrocatalogs/sne-2015-2019](https://github.com/astrocatalogs/sne-2015-2019): supernovae and supernova candidates discovered between 2015 and 2019.
- [astrocatalogs/sne-boneyard](https://github.com/astrocatalogs/sne-boneyard): transients that were initially flagged as possible supernovae but were later reclassified as something else and removed from the main catalog.

Each repository stores one JSON file per object, all following the same schema.

### How it is collected

The data is fetched live with Python using two endpoints that both return JSON:

1. GitHub's Git Trees API (`api.github.com/repos/.../git/trees/master?recursive=1`) returns a JSON listing of every file in a repository.
2. Each individual transient's data is then downloaded as JSON from GitHub's raw-content server (`raw.githubusercontent.com/...`).

### Cutting the data down

Both repositories are far larger than necessary for this project (12,677 files in `sne-2015-2019` and 1,988 in `sne-boneyard`). To keep the key questions answerable with a clean, balanced comparison, I've restricted the data to:

- discovery year 2019 only
- 2 discovery surveys (ZTF and ATLAS), aiming for 50 objects per survey, for up to 100 rows per repository (200 rows total).

The 50-per-survey target is a goal rather than a guarantee: the code below reports how many of each survey it actually found. In particular, `sne-boneyard` only contains about 46 ATLAS-discovered objects from 2019 in total, so that one group will fall a few rows short of 50 no matter how the data is sampled. A couple of filenames in `sne-boneyard` also contain stray HTML tags (a data-entry artifact from the original catalog import), which I handle in the download step by URL-encoding the filename.

In [1]:
import random

import pandas as pd
import requests
import urllib.parse

In [2]:
# --- Configuration ---------------------------------------------------------
OWNER = 'astrocatalogs'
BRANCH = 'master'

# only keep transients discovered in 2019
TARGET_YEAR = '2019'   
# the 2 discovery surveys we are comparing
SURVEYS = ['ZTF', 'ATLAS']   
# 50 x 2 surveys = 100 rows per repository
N_PER_SURVEY = 50               

# reproducible sample
random.seed(3000)  

In [3]:
def get_file_list(owner, repo, branch):
    # Call GitHub's Git Trees API (returns JSON) to list every .json file in the repo.
    url = f'https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    tree = response.json()['tree']
    return [item['path'] for item in tree if item['path'].endswith('.json')]


def fetch_record(session, owner, repo, branch, filename):
    # Download and parse one transient's JSON file from GitHub's raw file server.
    # Escape the few filenames that contain stray HTML tags.
    safe_name = urllib.parse.quote(filename)
    url = f'https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{safe_name}'
    try:
        response = session.get(url, timeout=15)
        if response.status_code != 200:
            return None
        record = response.json()
        name = next(iter(record))
        return record[name]
    except (requests.RequestException, ValueError, StopIteration):
        return None


def first_value(data, field):
    # most fields are lists of {'value': ..., 'source': ...} dicts; a few (like 'name') are plain strings.
    entries = data.get(field)
    if not entries:
        return None
    if isinstance(entries, str):
        return entries
    first = entries[0]
    return first.get('value') if isinstance(first, dict) else first

In [4]:
def collect_balanced_sample(session, owner, repo, branch, surveys, n_per_survey, target_year):
    # Scan a repository's files in random order, keeping only transients discovered in target_year whose discoverer is one of surveys, until each 
    # survey has n_per_survey rows (or every file has been checked).
    filenames = get_file_list(owner, repo, branch)
    random.shuffle(filenames)

    rows = []
    counts = {survey: 0 for survey in surveys}

    for i, filename in enumerate(filenames, start=1):
        if all(counts[survey] >= n_per_survey for survey in surveys):
            break

        data = fetch_record(session, owner, repo, branch, filename)
        if data is None:
            continue

        discoverdate = first_value(data, 'discoverdate')
        discoverer = first_value(data, 'discoverer')
        if not discoverdate or not discoverer:
            continue
        if not discoverdate.startswith(target_year):
            continue
        if discoverer not in surveys or counts[discoverer] >= n_per_survey:
            continue

        counts[discoverer] += 1
        rows.append({
            'name': first_value(data, 'name') or filename.replace('.json', ''),
            'discoverdate': discoverdate,
            'discoverer': discoverer,
            'claimedtype': first_value(data, 'claimedtype'),
            'maxappmag': first_value(data, 'maxappmag'),
            'n_photometry': len(data.get('photometry', [])),
            'repo': repo,
        })

        if i % 500 == 0:
            print(f'  ...checked {i}/{len(filenames)} files in {repo}, counts so far: {counts}')

    print(f'Finished {repo}: collected {len(rows)} rows -> {counts}')
    for survey in surveys:
        if counts[survey] < n_per_survey:
            print(f'  NOTE: only {counts[survey]} {survey} transients discovered in '
                  f'{target_year} exist in {repo} (fewer than the {n_per_survey} requested).')

    return pd.DataFrame(rows)

**Note:** the cell below makes one HTTP request per file it checks, so it can take a little while to run. However, as laid out in the rubric, it only needs to be run once; the result is stored in `sne_2019_sample.csv` afterward, so later analysis can just read the CSV.

In [5]:
session = requests.Session()

candidate_df = collect_balanced_sample(
    session, OWNER, 'sne-2015-2019', BRANCH, SURVEYS, N_PER_SURVEY, TARGET_YEAR
)
boneyard_df = collect_balanced_sample(
    session, OWNER, 'sne-boneyard', BRANCH, SURVEYS, N_PER_SURVEY, TARGET_YEAR
)

Finished sne-2015-2019: collected 100 rows -> {'ZTF': 50, 'ATLAS': 50}
Finished sne-boneyard: collected 96 rows -> {'ZTF': 50, 'ATLAS': 46}
  NOTE: only 46 ATLAS transients discovered in 2019 exist in sne-boneyard (fewer than the 50 requested).


In [6]:
# candidate/confirmed supernovae
candidate_df['is_false_positive'] = False 
# reclassified as NOT supernovae
boneyard_df['is_false_positive'] = True    

sne_df = pd.concat([candidate_df, boneyard_df], ignore_index=True)

# maxappmag arrives as a string (or is missing for some objects); make it numeric
sne_df['maxappmag'] = pd.to_numeric(sne_df['maxappmag'], errors='coerce')

# pull the discovery month out of the 'YYYY/MM/DD' discoverdate string
sne_df['discover_month'] = pd.to_datetime(
    sne_df['discoverdate'], format='%Y/%m/%d', errors='coerce'
).dt.month

print(sne_df.shape)
sne_df.head()

(196, 9)


,name,discoverdate,discoverer,claimedtype,maxappmag,n_photometry,repo,is_false_positive,discover_month
0,AT2019lee,2019/07/05,ZTF,Candidate,20.3,1,sne-2015-2019,False,7
1,AT2019csm,2019/03/25,ZTF,Candidate,19.7,5,sne-2015-2019,False,3
2,AT2019qsv,2019/09/23,ATLAS,Candidate,19.0,1,sne-2015-2019,False,9
3,AT2019dia,2019/04/01,ZTF,Candidate,19.2,1,sne-2015-2019,False,4
4,AT2019kur,2019/06/30,ATLAS,Candidate,19.8,1,sne-2015-2019,False,6


In [7]:
sne_df.to_csv('sne_2019_sample.csv', index=False)

In [8]:
print('Rows per repository:')
print(sne_df['repo'].value_counts())
print()

print('Rows per discovery survey:')
print(sne_df['discoverer'].value_counts())
print()

print('Rows per survey x real-vs-false-positive:')
print(sne_df.groupby(['discoverer', 'is_false_positive']).size())

Rows per repository:
repo
sne-2015-2019    100
sne-boneyard      96
Name: count, dtype: int64

Rows per discovery survey:
discoverer
ZTF      100
ATLAS     96
Name: count, dtype: int64

Rows per survey x real-vs-false-positive:
discoverer  is_false_positive
ATLAS       False                50
            True                 46
ZTF         False                50
            True                 50
dtype: int64


## 3. How the Data Will Be Used

Each row of `sne_2019_sample.csv` describes one transient discovered in 2019: two numeric features (`maxappmag`, its apparent brightness at discovery, and `n_photometry`, how many brightness measurements have been recorded for it) and several categorical features (`discoverer`, `claimedtype`, and the `is_false_positive` label marking whether the object came from the main candidate catalog or the boneyard). These map directly onto the two questions above. For Question 1, whether discovery-time features can separate real supernova candidates from false positives, `is_false_positive` is the class label I want to predict, and `maxappmag`, `n_photometry`, and `discoverer` are the predictor features, making this a classification problem, directly inspired by the real-bogus classifiers astronomers already build for surveys like ZTF (Duev et al., 2019). For Question 2, whether ZTF and ATLAS differ in how often their transients turn out to be false positives, I'll group the data by `discoverer` and compare the proportion of `is_false_positive == True` in each group, a relationship between two categorical variables that could later be tested formally or explored by including `discoverer` as a feature in the same classifier from Question 1 and inspecting how much it influences the prediction. Together, these let the project move from 'can we tell a real supernova candidate from a fake one?' to 'does the answer depend on which survey found it?'.